# Лабораторная работа IV. Классификация тональности отзывов Кинопоиска

**Датасет:** [Kinopoisk Movie Reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews) (Kaggle, папки `pos` / `neg` / `neu` с `.txt`-отзывами)  
**Задача:** классификация тональности (позитивный / негативный / нейтральный отзыв)  
**Фреймворк:** scikit-learn + CatBoost + Optuna  

В работе построен полный pipeline обработки текста и классификации:
подготовка Kaggle-датасета в `LAB_IV/data/dataset`, сборка рабочей таблицы из `.txt`-файлов,
разведочный анализ с Plotly-графиками и облаками слов,
предобработка (нижний регистр → пунктуация → токенизация → стоп-слова → лемматизация spaCy),
подготовка данных к обучению (train/test split, TF-IDF векторизация),
сравнение трёх обязательных baseline-моделей (LogisticRegression, LinearSVC, CatBoostClassifier),
подбор гиперпараметров через Optuna, анализ ошибок
и тест на пяти новых текстах в разных стилях.


## Оглавление

1. [Подготовка окружения](#Подготовка-окружения)
2. [Загрузка данных](#Загрузка-данных)
3. [Разведочный анализ данных](#Разведочный-анализ-данных)
4. [Предобработка текстов](#Предобработка-текстов)
5. [Подготовка данных к обучению](#Подготовка-данных-к-обучению)
6. [Классификация](#Классификация)
7. [Анализ ошибок](#Анализ-ошибок)
8. [Тест на новых текстах](#Тест-на-новых-текстах)
9. [Выводы](#Выводы)

## Подготовка окружения

Первый блок фиксирует воспроизводимость, подключает все нужные библиотеки,
задаёт режим обучения CPU/CUDA для CatBoost и загружает языковую модель spaCy
для русского языка. Фиксация `SEED=42` обеспечивает одинаковые результаты
при повторном запуске.


### 1.1 Импорты и глобальные настройки

Сначала подключаем библиотеки и проверяем обязательные зависимости. Отдельной короткой ячейкой ниже фиксируем `SEED`, режим предупреждений и verbosity Optuna, чтобы дальше код не смешивал импорты с настройками запуска.


In [1]:
import json
import os
import re
import random
import shutil
import warnings
from collections import Counter
from pathlib import Path

from joblib import dump, load
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud

try:
    import spacy
    from spacy.lang.ru.stop_words import STOP_WORDS as SPACY_RU_STOPWORDS
except ImportError as exc:
    raise RuntimeError("spaCy не установлен. Выполните: %pip install -q spacy") from exc

try:
    import optuna
except ImportError as exc:
    raise RuntimeError("Optuna обязательна для подбора гиперпараметров. Выполните: %pip install -q optuna") from exc

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise RuntimeError("CatBoost обязателен для этой работы. Выполните: %pip install -q catboost") from exc

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score,
)

from tqdm import tqdm
from IPython.display import display, Markdown


### 1.1.1 Воспроизводимость и служебные настройки

Фиксируем случайность, приглушаем лишние предупреждения и делаем короткую проверку, что базовый блок окружения собрался без ошибок.


In [2]:
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed()
print("Imports OK")


Imports OK


### 1.2 Переключатель CPU/CUDA для обучения

Сначала задаём желаемый режим устройства, затем отдельными ячейками проверяем CUDA и собираем параметры CatBoost. `DEVICE_MODE = "auto"` остаётся безопасным режимом: на CUDA-машине модель уйдёт на GPU, а в обычном окружении останется CPU.


In [3]:
DEVICE_MODE = "auto"  # варианты: "auto", "cpu", "cuda"
CATBOOST_CUDA_DEVICES = "0"  # например: "0" или "0:1" для нескольких GPU


### 1.2.1 Проверка доступности CUDA

Helper-функции аккуратно проверяют PyTorch/CUDA и переводят выбранный режим в параметры, которые понимает CatBoost.


In [4]:
def get_cuda_status() -> tuple[bool, str]:
    try:
        import torch
    except Exception as exc:
        return False, f"torch недоступен: {type(exc).__name__}: {exc}"

    try:
        cuda_available = bool(torch.cuda.is_available())
    except Exception as exc:
        return False, f"torch.cuda.is_available() завершился с ошибкой: {type(exc).__name__}: {exc}"

    torch_cuda_version = getattr(getattr(torch, "version", None), "cuda", None)
    status = f"torch={getattr(torch, '__version__', 'unknown')}, torch CUDA={torch_cuda_version}"
    return cuda_available, status


def is_cuda_available() -> bool:
    cuda_available, _ = get_cuda_status()
    return cuda_available


def resolve_training_device(mode: str = DEVICE_MODE) -> dict:
    normalized_mode = str(mode).strip().lower()
    allowed_modes = {"auto", "cpu", "cuda"}
    if normalized_mode not in allowed_modes:
        raise ValueError(f"DEVICE_MODE должен быть одним из {sorted(allowed_modes)}, получено: {mode!r}")

    cuda_available, cuda_status = get_cuda_status()
    use_cuda = normalized_mode == "cuda" or (normalized_mode == "auto" and cuda_available)

    if normalized_mode == "cuda" and not cuda_available:
        raise RuntimeError(
            "Выбран DEVICE_MODE='cuda', но CUDA недоступна в текущем окружении. "
            "Проверьте CUDA/PyTorch на GPU-машине или переключите DEVICE_MODE на 'auto'/'cpu'. "
            f"Статус: {cuda_status}"
        )

    catboost_task_type = "GPU" if use_cuda else "CPU"
    catboost_params = {"task_type": catboost_task_type}
    if use_cuda:
        catboost_params["devices"] = CATBOOST_CUDA_DEVICES

    return {
        "requested_mode": normalized_mode,
        "cuda_available": cuda_available,
        "cuda_status": cuda_status,
        "catboost_task_type": catboost_task_type,
        "catboost_params": catboost_params,
    }


### 1.2.2 Итоговое устройство обучения

Разрешаем выбранный режим в `CATBOOST_DEVICE_PARAMS` и печатаем, где фактически будет обучаться CatBoost.


In [5]:
TRAINING_DEVICE = resolve_training_device()
CATBOOST_TASK_TYPE = TRAINING_DEVICE["catboost_task_type"]
CATBOOST_DEVICE_PARAMS = TRAINING_DEVICE["catboost_params"]

print("Requested device mode:", TRAINING_DEVICE["requested_mode"])
print("CUDA available:", TRAINING_DEVICE["cuda_available"])
print("CUDA status:", TRAINING_DEVICE["cuda_status"])
print("CatBoost task_type:", CATBOOST_TASK_TYPE)
if CATBOOST_TASK_TYPE == "GPU":
    print("CatBoost CUDA devices:", CATBOOST_CUDA_DEVICES)


Requested device mode: auto
CUDA available: True
CUDA status: torch=2.11.0+cu126, torch CUDA=12.6
CatBoost task_type: GPU
CatBoost CUDA devices: 0


### 1.3 Функции визуализации Plotly

Визуализации разбиты на маленькие группы: общая тема и цвета, EDA-графики, частотные графики и матрица ошибок. Так дальше в ноутбуке остаются только короткие вызовы готовых функций.


In [6]:
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = px.colors.qualitative.Set2

BASIC_SW = {
    "и", "в", "на", "что", "это", "как", "с", "из", "а", "но",
    "по", "за", "то", "так", "же", "был", "была", "всё", "он",
    "она", "они", "или", "к", "у", "от", "для", "его", "её",
    "их", "ещё", "уже", "о", "об", "при", "до", "со", "чем",
    "во", "бы", "ли", "тут", "тоже", "всех", "этот", "этого",
}


def show_plotly_figure(fig, title: str | None = None, *, height: int = 480):
    '''Apply the shared notebook style and render a Plotly figure.'''
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=title,
        title_x=0.02,
        height=height,
        margin=dict(l=40, r=30, t=75 if title else 45, b=40),
        legend_title_text="Класс",
        font=dict(size=12),
    )
    fig.show()


def make_class_color_map(labels) -> dict:
    labels = [str(label) for label in labels]
    return {label: PLOTLY_COLORS[i % len(PLOTLY_COLORS)] for i, label in enumerate(labels)}


### 1.3.1 Распределения классов и длин

Функция строит пару графиков для EDA: число отзывов по классам и распределение длин текстов.


In [7]:
def plot_class_counts_and_lengths(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    word_count_col: str = "word_count",
):
    counts = data[target_col].value_counts()
    color_map = make_class_color_map(counts.index)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Распределение классов", "Распределение длин отзывов"),
    )
    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            text=[f"{value:,}" for value in counts.values],
            textposition="outside",
            marker_color=[color_map[str(label)] for label in counts.index],
            name="Количество отзывов",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    for label in counts.index:
        subset = data.loc[data[target_col] == label, word_count_col]
        fig.add_trace(
            go.Histogram(
                x=subset,
                nbinsx=60,
                opacity=0.55,
                marker_color=color_map[str(label)],
                name=str(label),
            ),
            row=1,
            col=2,
        )

    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text="Класс", row=1, col=1)
    fig.update_yaxes(title_text="Количество отзывов", row=1, col=1)
    fig.update_xaxes(title_text="Количество слов", range=[0, 500], row=1, col=2)
    fig.update_yaxes(title_text="Частота", row=1, col=2)
    return show_plotly_figure(fig, "Базовая статистика датасета", height=500)


### 1.3.2 WordCloud и топ слов

Здесь собраны функции для облаков слов и топов частот. Это ещё сырой текст, поэтому стоп-слова фильтруются только грубо.


In [8]:
def plot_wordclouds_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    max_words: int = 150,
):
    classes = data[target_col].dropna().unique().tolist()
    cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "Greys"]
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Облако слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        text = " ".join(data.loc[data[target_col] == label, text_col].astype(str))
        wc = WordCloud(
            width=700,
            height=420,
            background_color="white",
            colormap=cmaps[(col_idx - 1) % len(cmaps)],
            max_words=max_words,
            collocations=False,
        ).generate(text)
        fig.add_trace(go.Image(z=wc.to_array(), hoverinfo="skip"), row=1, col=col_idx)
        fig.update_xaxes(visible=False, row=1, col=col_idx)
        fig.update_yaxes(visible=False, row=1, col=col_idx)

    return show_plotly_figure(
        fig,
        "Наиболее частотные слова по классам (до предобработки)",
        height=500,
    )


def get_top_words(texts, *, n: int = 20, stopwords: set[str] | None = None):
    stopwords = BASIC_SW if stopwords is None else stopwords
    words = [
        word.lower()
        for text in texts
        for word in re.findall(r"[а-яёА-ЯЁ]+", str(text))
        if word.lower() not in stopwords and len(word) > 2
    ]
    return Counter(words).most_common(n)


def plot_top_words_by_class(
    data: pd.DataFrame,
    *,
    target_col: str = "sentiment",
    text_col: str = "review",
    n: int = 20,
):
    classes = data[target_col].dropna().unique().tolist()
    color_map = make_class_color_map(classes)
    fig = make_subplots(
        rows=1,
        cols=len(classes),
        subplot_titles=[f"Топ-{n} слов — {str(label).capitalize()}" for label in classes],
    )

    for col_idx, label in enumerate(classes, start=1):
        top = get_top_words(data.loc[data[target_col] == label, text_col], n=n)
        if not top:
            continue
        words, freqs = zip(*top)
        fig.add_trace(
            go.Bar(
                x=list(freqs)[::-1],
                y=list(words)[::-1],
                orientation="h",
                marker_color=color_map[str(label)],
                name=str(label),
                showlegend=False,
            ),
            row=1,
            col=col_idx,
        )
        fig.update_xaxes(title_text="Частота", row=1, col=col_idx)

    return show_plotly_figure(fig, f"Топ-{n} слов по классам", height=580)


### 1.3.3 Матрица ошибок

Отдельная функция рисует confusion matrix в едином Plotly-оформлении.


In [9]:
def plot_confusion_matrix_plotly(
    y_true,
    y_pred,
    labels,
    *,
    title: str = "Матрица ошибок",
):
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    matrix_df = pd.DataFrame(matrix, index=labels, columns=labels)
    fig = px.imshow(
        matrix_df,
        text_auto=True,
        color_continuous_scale="Blues",
        aspect="auto",
        labels=dict(x="Предсказано", y="Истинный класс", color="Количество"),
    )
    fig.update_traces(hovertemplate="Истинный класс: %{y}<br>Предсказано: %{x}<br>Количество: %{z}<extra></extra>")
    return show_plotly_figure(fig, title, height=520)


### 1.4 Загрузка языковой модели spaCy

`ru_core_news_sm` — компактная модель для русского языка.
Используется исключительно для лемматизации:
`disable=["parser", "ner"]` оставляет только `tok2vec` + `morphologizer` + `lemmatizer`,
что ускоряет обработку примерно в 3 раза по сравнению с полным конвейером.

Если модель не установлена, выполните в терминале:
```
python -m spacy download ru_core_news_sm
```


In [10]:
SPACY_MODEL = "ru_core_news_sm"

try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
    print("spaCy model loaded:", nlp.meta["name"])
except OSError as exc:
    raise RuntimeError(
        f"Модель spaCy {SPACY_MODEL!r} не найдена.\n"
        "Установите зависимости в отдельной ячейке и перезапустите kernel:\n"
        "%pip install -q spacy\n"
        f"!python -m spacy download {SPACY_MODEL}"
    ) from exc

spaCy model loaded: core_news_sm


Окружение готово. Модель `ru_core_news_sm` загружена без NER и парсера —
нам нужна только лемматизация.


## Загрузка данных

Датасет Kinopoisk Movie Reviews с Kaggle хранится не как один CSV-файл:
в архиве лежат директории классов `pos`, `neg`, `neu`, а каждый отзыв находится
в отдельном `.txt`-файле. В этом разделе приводим такой формат к рабочей таблице
с колонками `review` и `sentiment`.


### 2.1 Подготовка Kaggle-датасета и первичный осмотр

Блок разбит на четыре шага: пути и константы, поиск исходных папок `pos`/`neg`/`neu`, чтение `.txt`-файлов и финальная загрузка prepared-таблицы из кэша или пересборка. Повторные запуски не перечитывают датасет без причины.


In [11]:
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"
KAGGLE_DATASET = "mikhailklemin/kinopoisks-movies-reviews"
LABEL_DIR_MAP = {
    "neg": "negative",
    "neu": "neutral",
    "pos": "positive",
}


def find_lab_iv_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if base.name == "LAB_IV" and (base / "data").exists():
            return base
        lab_candidate = base / "LAB_IV"
        if (lab_candidate / "data").exists():
            return lab_candidate
    raise FileNotFoundError("Не найдена директория LAB_IV/data относительно текущего пути запуска")


LAB_IV_DIR = find_lab_iv_dir()
DATA_DIR = LAB_IV_DIR / "data"
DATASET_DIR = DATA_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME


### 2.1.1 Поиск и подготовка raw-датасета

Helper-функции находят папки классов, при необходимости скачивают Kaggle-датасет и приводят локальную структуру к одному виду.


In [12]:
def count_txt_files(root: Path) -> int:
    return sum(1 for _ in root.rglob("*.txt")) if root.exists() else 0


def get_review_dir_counts(root: Path) -> dict[str, int]:
    return {
        label_dir: count_txt_files(root / label_dir)
        for label_dir in LABEL_DIR_MAP
    }


def has_review_dirs(root: Path) -> bool:
    return all((root / label_dir).is_dir() for label_dir in LABEL_DIR_MAP)


def find_review_dirs_root(root: Path) -> Path | None:
    root = Path(root)
    if has_review_dirs(root):
        return root
    if root.exists():
        for candidate in root.rglob("*"):
            if candidate.is_dir() and has_review_dirs(candidate):
                return candidate
    return None


def copy_review_dirs(source_root: Path, target_root: Path) -> None:
    source_root = source_root.resolve()
    target_root = target_root.resolve()
    if source_root == target_root:
        return

    target_root.mkdir(parents=True, exist_ok=True)
    for label_dir in LABEL_DIR_MAP:
        src = source_root / label_dir
        dst = target_root / label_dir
        dst.mkdir(parents=True, exist_ok=True)
        for src_file in src.rglob("*.txt"):
            rel_path = src_file.relative_to(src)
            dst_file = dst / rel_path
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst_file)


def download_kaggle_dataset() -> Path:
    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError(
            "Папки pos/neg/neu не найдены и не установлен kagglehub. "
            "Установите зависимости через uv sync --dev или добавьте kagglehub вручную."
        ) from exc

    errors: list[str] = []
    download_attempts = [{"output_dir": str(DATASET_DIR)}, {}]
    for kwargs in download_attempts:
        try:
            return Path(kagglehub.dataset_download(KAGGLE_DATASET, **kwargs))
        except TypeError as exc:
            errors.append(f"dataset_download({kwargs}) не поддерживается: {exc}")
        except Exception as exc:
            errors.append(f"dataset_download({kwargs}) завершился ошибкой: {exc}")

    raise RuntimeError("Не удалось скачать Kaggle-датасет. Ошибки: " + " | ".join(errors))


def ensure_raw_dataset() -> Path:
    local_root = find_review_dirs_root(DATASET_DIR)
    if local_root is not None:
        copy_review_dirs(local_root, DATASET_DIR)
        return DATASET_DIR

    downloaded_dir = download_kaggle_dataset()
    downloaded_root = find_review_dirs_root(downloaded_dir)
    if downloaded_root is None:
        raise FileNotFoundError(
            "В скачанном Kaggle-датасете не найдены папки pos/neg/neu. "
            f"Проверенный путь: {downloaded_dir.resolve()}"
        )

    copy_review_dirs(downloaded_root, DATASET_DIR)
    return DATASET_DIR


### 2.1.2 Сборка таблицы из `.txt`

Каждый файл читается с запасными кодировками, из имени достаются `movie_id` и `review_id`, а затем всё собирается в один `DataFrame`.


In [13]:
def read_review_text(path: Path) -> str:
    for encoding in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return path.read_text(encoding=encoding).strip()
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace").strip()


def parse_review_ids(path: Path) -> tuple[int | None, int | None]:
    parts = path.stem.split("-", maxsplit=1)
    if len(parts) != 2:
        return None, None
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return None, None


def build_reviews_dataframe(raw_root: Path) -> pd.DataFrame:
    records = []
    total_files = sum(get_review_dir_counts(raw_root).values())
    progress = tqdm(
        total=total_files,
        desc="Reading review txt files",
        dynamic_ncols=True,
        mininterval=1.0,
        leave=False,
    )

    with progress:
        for label_dir, sentiment in LABEL_DIR_MAP.items():
            label_root = raw_root / label_dir
            for txt_path in sorted(label_root.rglob("*.txt")):
                movie_id, review_id = parse_review_ids(txt_path)
                records.append({
                    "review": read_review_text(txt_path),
                    "sentiment": sentiment,
                    "source_label_dir": label_dir,
                    "source_file": str(txt_path.relative_to(raw_root)),
                    "movie_id": movie_id,
                    "review_id": review_id,
                })
                progress.update(1)

    if not records:
        raise ValueError(f"В {raw_root.resolve()} не найдено ни одного .txt-отзыва")

    result = pd.DataFrame.from_records(records)
    result["movie_id"] = result["movie_id"].astype("Int64")
    result["review_id"] = result["review_id"].astype("Int64")
    return result


### 2.1.3 Prepared-кэш

Если `kinopoisk_reviews_prepared.csv` совпадает с raw-датасетом по размеру и колонкам, берём его. Иначе пересобираем таблицу заново.


In [14]:
def load_or_build_reviews_dataframe() -> tuple[pd.DataFrame, Path, dict[str, int]]:
    raw_dataset_root = ensure_raw_dataset()
    raw_counts = get_review_dir_counts(raw_dataset_root)
    raw_total = sum(raw_counts.values())

    if raw_total == 0:
        raise ValueError(f"Папки pos/neg/neu найдены в {raw_dataset_root.resolve()}, но .txt-файлов в них нет")

    cache_is_fresh = False
    if DATA_PATH.exists():
        cached_df = pd.read_csv(DATA_PATH)
        has_required_columns = {"review", "sentiment"}.issubset(cached_df.columns)
        cache_is_fresh = has_required_columns and len(cached_df) == raw_total
        if cache_is_fresh:
            df = cached_df
            print("Loaded prepared table:", DATA_PATH.resolve())
        else:
            print("Prepared table exists, but does not match raw dataset. Rebuilding cache...")

    if not cache_is_fresh:
        df = build_reviews_dataframe(raw_dataset_root)
        df.to_csv(DATA_PATH, index=False)
        print("Prepared table saved:", DATA_PATH.resolve())

    return df, raw_dataset_root, raw_counts


df, raw_dataset_root, raw_counts = load_or_build_reviews_dataframe()

print("Dataset directory:", DATASET_DIR.resolve())
print("Raw review counts:", raw_counts)
print("Prepared table:", DATA_PATH.resolve())
print("Prepared shape:", df.shape)


Loaded prepared table: D:\DEV\ML2\LAB_IV\data\dataset\kinopoisk_reviews_prepared.csv
Dataset directory: D:\DEV\ML2\LAB_IV\data\dataset
Raw review counts: {'neg': 19827, 'neu': 24704, 'pos': 87138}
Prepared table: D:\DEV\ML2\LAB_IV\data\dataset\kinopoisk_reviews_prepared.csv
Prepared shape: (131669, 6)


### 2.1.4 Быстрый осмотр таблицы

Печатаем путь к prepared-файлу, первые строки и распределение классов, чтобы сразу проверить форму данных глазами.


In [15]:
print("Prepared data path:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
df.info()


Prepared data path: D:\DEV\ML2\LAB_IV\data\dataset\kinopoisk_reviews_prepared.csv
Shape: (131669, 6)
Columns: ['review', 'sentiment', 'source_label_dir', 'source_file', 'movie_id', 'review_id']


,review,sentiment,source_label_dir,source_file,movie_id,review_id
0,В 2003-ем году под руководством малоизвестного...,negative,neg,neg\1000083-0.txt,1000083,0
1,"Грустно и печально. Грустно от того, что довол...",negative,neg,neg\1000083-1.txt,1000083,1
2,Давным-давно Кира Найтли ворвалась на экран от...,negative,neg,neg\1000125-3.txt,1000125,3
3,"Я, в общем, ничего против уравновешенного феми...",negative,neg,neg\1000125-4.txt,1000125,4
4,"Измена — один из сюжетов, который всегда будет...",negative,neg,neg\1000125-6.txt,1000125,6


<class 'pandas.DataFrame'>
RangeIndex: 131669 entries, 0 to 131668
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   review            131669 non-null  str  
 1   sentiment         131669 non-null  str  
 2   source_label_dir  131669 non-null  str  
 3   source_file       131669 non-null  str  
 4   movie_id          131669 non-null  int64
 5   review_id         131669 non-null  int64
dtypes: int64(2), str(4)
memory usage: 523.7 MB


### 2.2 Нормализация столбцов и меток

После сборки из `.txt`-файлов таблица уже содержит `review` и `sentiment`.
Дополнительно нормализуем метки классов, удаляем строки с пустым текстом или
отсутствующей меткой и сохраняем служебные поля `source_file`, `movie_id`,
`review_id` для последующего анализа ошибок.


In [16]:
required_columns = {"review", "sentiment"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"В подготовленной таблице нет обязательных колонок: {sorted(missing_columns)}")

metadata_columns = [
    column
    for column in ["source_label_dir", "source_file", "movie_id", "review_id"]
    if column in df.columns
]

df = df[["review", "sentiment", *metadata_columns]].copy()

# Normalize sentiment labels to lowercase strings
label_map = {
    "1": "positive", "0": "negative", "-1": "negative",
    "good": "positive", "bad": "negative",
    "pos": "positive", "neg": "negative", "neu": "neutral",
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "позитивный": "positive", "негативный": "negative", "нейтральный": "neutral",
}

df["sentiment"] = (
    df["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(label_map)
)

# Drop nulls and empty reviews
df.dropna(subset=["review", "sentiment"], inplace=True)
df["review"] = df["review"].astype(str)
df = df[df["review"].str.strip().astype(bool)].reset_index(drop=True)

expected_labels = set(LABEL_DIR_MAP.values())
unexpected_labels = sorted(set(df["sentiment"]) - expected_labels)
if unexpected_labels:
    print("Warning: unexpected labels after normalization:", unexpected_labels)

print("Shape after cleaning:", df.shape)
print("Unique sentiments:", sorted(df["sentiment"].unique()))
print(df["sentiment"].value_counts())


Shape after cleaning: (131669, 6)
Unique sentiments: ['negative', 'neutral', 'positive']
sentiment
positive    87138
neutral     24704
negative    19827
Name: count, dtype: int64


> **Вывод:**

Датасет загрузился без потерь: всего прочитано `131669` `.txt`-отзывов, из них `87138` из `pos`, `24704` из `neu` и `19827` из `neg`. После нормализации таблица сохранила размер `(131669, 6)`, все основные поля заполнены, а пустые отзывы после очистки не появились.

Распределение классов сразу показывает главную особенность задачи: `positive` сильно доминирует над `neutral` и `negative`. В целом данные пригодны для классификации тональности, но дальше нельзя ориентироваться только на `accuracy`, потому что модель может неплохо выглядеть просто за счет частого положительного класса.


## Разведочный анализ данных

Цель EDA — понять распределение классов, характер текстов и наиболее
информативные слова каждого класса. Суммарно не более 8 графиков.


### 3.1 Распределение классов и длины текстов

Два графика в одной строке:
столбчатая диаграмма числа отзывов по классам
и гистограмма длин текстов (в словах) с разбивкой по классам.
Это показывает, есть ли дисбаланс классов
и насколько тексты однородны по длине.


In [17]:
df["word_count"] = df["review"].str.split().str.len()
plot_class_counts_and_lengths(df)

print(df.groupby("sentiment")["word_count"].describe().round(1))


             count   mean    std   min    25%    50%    75%     max
sentiment                                                          
negative   19827.0  344.5  199.3  10.0  200.0  298.0  440.0  1087.0
neutral    24704.0  329.2  198.3   9.0  186.0  285.0  427.0  1756.0
positive   87138.0  336.1  201.7   8.0  190.0  287.0  429.0  2059.0


> **Вывод:**

Самый частый класс — `positive` (`87138` отзывов), самый редкий — `negative` (`19827` отзывов). Разница между ними примерно в `4.4` раза, поэтому `accuracy` будет довольно снисходительной метрикой: можно часто угадывать позитив и получать неплохое качество, но при этом плохо работать с редкими классами. Поэтому `macro_f1` здесь полезнее, он честнее наказывает за слабые `neutral` и `negative`.

По длинам классы похожи: средняя длина держится около `329-345` слов, медиана около `285-298`. При этом есть длинные выбросы до `2059` слов, но это скорее особенность отзывов Кинопоиска, а не ошибка данных. Для обучения я считаю правильным использовать `class_weight` / `auto_class_weights`, иначе модель почти неизбежно будет тянуться к `positive`.


### 3.2 Облака слов по классам

WordCloud визуализирует наиболее частотные слова каждого класса на сырых текстах
(до предобработки). Это даёт интуитивное представление о лексиконе классов.


In [18]:
plot_wordclouds_by_class(df)


> **Вывод:**

Облака слов показывают, что на сырых текстах во всех классах доминирует киношная общая лексика: `фильм`, `фильма`, `все`, `очень`, `просто`, `можно`, `кино`. Для `positive` сильнее заметны частые слова вроде `очень` и `все`, для `negative` дополнительно видны `нет`, `даже`, `только`, а `neutral` визуально оказывается ближе к смеси двух остальных классов.

Главный практический вывод такой: лексический сигнал есть, но на облаках он забит слишком общими словами. Без нормальной предобработки и TF-IDF модель будет видеть не столько тональность, сколько сам факт того, что это отзыв о фильме.


### 3.3 Топ-20 слов по классам

Гистограмма частот 20 самых популярных слов для каждого класса
после грубой фильтрации служебных частей речи через `Counter`.
Это количественная картина поверх облаков слов.

> Слово «не» **намеренно оставлено** в обоих наборах —
> оно несёт семантическую нагрузку в негативных отзывах.


In [19]:
plot_top_words_by_class(df)


> **Вывод:**

Топ-20 подтверждает картину из облаков: во всех классах наверху стоят `фильм`, `все`, `фильма`, `очень`, `просто`, `если`, `который`, `кино`. В `negative` заметно слово `нет` (`16478` вхождений), а в `positive` особенно часто встречается `очень` (`108476`), но в целом пересечение топов слишком большое, чтобы по ним одним отделить тональность.

Частицу `не` и слово `нет` важно не выкидывать: для отзывов это не просто служебные слова, а переключатели смысла. Если удалить отрицание, фразы вроде `не понравилось` или `не отвечает требованиям` могут стать почти положительными по набору оставшихся слов, и это как раз тот случай, где простая очистка может испортить модель.


### 3.4 Общая статистика словаря

Считаем суммарный объём словаря, среднюю и медианную длину отзыва.
Эта числовая сводка дополняет визуализации.


In [20]:
all_words = [
    w.lower() for text in df["review"].astype(str)
    for w in re.findall(r"[а-яёА-ЯЁ]+", text)
]
print(f"Всего токенов:      {len(all_words):>10,}")
print(f"Уникальных токенов: {len(set(all_words)):>10,}")
print(f"\nСредняя длина отзыва: {df['word_count'].mean():.1f} слов")
print(f"Медиана длины:        {df['word_count'].median():.1f} слов")
print(f"Максимальная длина:   {df['word_count'].max()} слов")


Всего токенов:      43,745,399
Уникальных токенов:    608,998

Средняя длина отзыва: 336.0 слов
Медиана длины:        288.0 слов
Максимальная длина:   2059 слов


> **Вывод:**

Общий объем текста получился большой: `43,745,399` токенов и `608,998` уникальных токенов. Средний отзыв содержит `336.0` слов, медианный — `288.0`, а максимум доходит до `2059` слов. То есть датасет не маленький, но словарь ожидаемо шумный: в отзывах много имен, названий, редких форм и просто авторской лексики.

`max_features=50_000` для baseline TF-IDF выглядит разумным компромиссом. Он не покрывает весь уникальный словарь, зато отсекает длинный хвост редких слов и оставляет достаточно признаков для линейных моделей. Я бы не стал сразу расширять словарь без проверки качества, потому что размерность и так уже довольно высокая.


## Предобработка текстов

Предобработка приводит тексты к нормализованному виду:
нижний регистр → удаление пунктуации и цифр → лемматизация spaCy →
удаление стоп-слов (кроме «не» и «нет»).
Результат сохраняется в столбце `cleaned_text`.


### 4.1 Набор стоп-слов

Используем встроенный список spaCy для русского языка,
но исключаем «не» и «нет» — они принципиально меняют смысл высказывания
(ср. «хороший» vs «не хороший»).


In [21]:
STOPWORDS = set(SPACY_RU_STOPWORDS)
STOPWORDS.discard("не")
STOPWORDS.discard("нет")

print(f"Стоп-слов в наборе: {len(STOPWORDS)}")
print(f"'не' в стоп-словах: {'не' in STOPWORDS}")
print(f"'нет' в стоп-словах: {'нет' in STOPWORDS}")


Стоп-слов в наборе: 766
'не' в стоп-словах: False
'нет' в стоп-словах: False


### 4.2 Функция предобработки одного текста

Шаги:
1. Нижний регистр.
2. Удаление пунктуации и цифр (оставляем только кириллицу и пробелы).
3. Лемматизация через spaCy: каждый токен заменяется нормальной формой.
4. Удаление стоп-слов и коротких токенов (len < 2).


In [22]:
def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яёa-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_doc(doc) -> str:
    return " ".join(
        token.lemma_ for token in doc
        if token.lemma_ not in STOPWORDS
        and len(token.lemma_) >= 2
        and not token.is_space
    )


def preprocess_text(text: str) -> str:
    return clean_doc(nlp(normalize_text(text)))


def preprocess_texts(texts, batch_size: int = 256, n_process: int = 1) -> list[str]:
    normalized_texts = [normalize_text(text) for text in texts]
    docs = nlp.pipe(normalized_texts, batch_size=batch_size, n_process=n_process)
    return [
        clean_doc(doc)
        for doc in tqdm(
            docs,
            total=len(normalized_texts),
            desc="Preprocessing",
            dynamic_ncols=True,
            mininterval=1.0,
            leave=False,
        )
    ]


# Smoke test
sample = "Этот фильм не оставил меня равнодушным! Просто великолепно."
print("Вход: ", sample)
print("Выход:", preprocess_text(sample))

Вход:  Этот фильм не оставил меня равнодушным! Просто великолепно.
Выход: фильм не оставить равнодушный великолепно


> **Вывод:**

Smoke-test отработал так, как и хотелось: фраза `Этот фильм не оставил меня равнодушным! Просто великолепно.` превратилась в `фильм не оставить равнодушный великолепно`. Частица `не` сохранилась, пунктуация и лишние символы ушли, а слова стали ближе к нормальной форме: `оставил` → `оставить`, `равнодушным` → `равнодушный`.

Перед обучением на полном датасете важно было проверить, не появляются ли пустые `cleaned_text` и не слишком ли агрессивно стоп-слова вычищают смысл. Особенно это касается отрицаний, потому что для тональности они важнее многих обычных частотных слов.


### 4.3 Применение к датасету

Предобработка вынесена в кэшируемый блок: отдельно задаём параметры и пути, отдельно проверяем пригодность кэша, а в финальной ячейке либо загружаем `cleaned_text`, либо запускаем долгий проход spaCy.


In [23]:
PREPROCESS_BATCH_SIZE = 256
PREPROCESS_N_PROCESS = min(4, os.cpu_count() or 1)
PREPROCESSING_VERSION = "spacy-lemma-v1"

CLEANED_PARQUET_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.parquet")
CLEANED_CSV_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.csv")
CACHE_META_PATH = DATA_PATH.with_name("kinopoisk_reviews_cleaned.meta.json")
CACHE_META_KEYS = (
    "spacy_model",
    "spacy_model_version",
    "preprocessing_version",
    "source_rows",
    "source_size_bytes",
    "review_chars_total",
    "stopwords",
)


### 4.3.1 Проверка кэша предобработки

Метаданные фиксируют версию spaCy, размер исходных данных и набор стоп-слов. Это защищает от ситуации, когда старый `cleaned_text` случайно используется после изменения очистки.


In [24]:
def build_cache_metadata(source_df: pd.DataFrame) -> dict:
    return {
        "spacy_model": SPACY_MODEL,
        "spacy_model_version": nlp.meta.get("version"),
        "preprocessing_version": PREPROCESSING_VERSION,
        "source_rows": int(len(source_df)),
        "source_path": str(DATA_PATH.resolve()),
        "source_size_bytes": int(DATA_PATH.stat().st_size) if DATA_PATH.exists() else None,
        "review_chars_total": int(source_df["review"].fillna("").str.len().sum()),
        "stopwords": sorted(STOPWORDS),
    }


def cache_frame_is_usable(cached_df: pd.DataFrame, source_df: pd.DataFrame) -> bool:
    if "cleaned_text" not in cached_df.columns:
        return False
    if len(cached_df) != len(source_df):
        return False

    required_columns = [column for column in ("review", "sentiment") if column in source_df.columns]
    return all(column in cached_df.columns for column in required_columns)


def cache_metadata_matches(expected_meta: dict) -> bool:
    if not CACHE_META_PATH.exists():
        return False

    try:
        with CACHE_META_PATH.open("r", encoding="utf-8") as cache_meta_file:
            actual_meta = json.load(cache_meta_file)
    except (OSError, json.JSONDecodeError):
        return False

    return all(actual_meta.get(key) == expected_meta.get(key) for key in CACHE_META_KEYS)


def write_cache_metadata(meta: dict) -> None:
    with CACHE_META_PATH.open("w", encoding="utf-8") as cache_meta_file:
        json.dump(meta, cache_meta_file, ensure_ascii=False, indent=2)


def read_cleaned_cache(cache_path: Path) -> pd.DataFrame:
    if cache_path.suffix == ".parquet":
        return pd.read_parquet(cache_path)
    return pd.read_csv(cache_path)


def find_cleaned_cache(source_df: pd.DataFrame, expected_meta: dict):
    for cache_path in (CLEANED_PARQUET_PATH, CLEANED_CSV_PATH):
        if not cache_path.exists():
            continue

        print(f"Found cleaned cache: {cache_path.resolve()}")
        cached_df = read_cleaned_cache(cache_path)
        if not cache_frame_is_usable(cached_df, source_df):
            print("Cache schema or row count does not match. Rebuilding...")
            continue

        if cache_metadata_matches(expected_meta):
            return cached_df, cache_path

        if not CACHE_META_PATH.exists():
            print("Cache metadata is missing; using cache after basic shape check.")
            write_cache_metadata(expected_meta)
            return cached_df, cache_path

        print("Cache metadata does not match current preprocessing. Rebuilding...")

    return None, None


### 4.3.2 Загрузка или пересчёт `cleaned_text`

Если кэш валиден, берём его. Если нет — обрабатываем отзывы через `nlp.pipe`, сохраняем Parquet и показываем несколько примеров.


In [25]:
def load_or_preprocess_reviews(source_df: pd.DataFrame) -> pd.DataFrame:
    expected_cache_meta = build_cache_metadata(source_df)
    cached_df, cache_path = find_cleaned_cache(source_df, expected_cache_meta)

    if cached_df is not None:
        print(f"Loaded cleaned data from cache → {cache_path.resolve()}")
        if cache_path != CLEANED_PARQUET_PATH:
            cached_df.to_parquet(CLEANED_PARQUET_PATH, index=False)
            write_cache_metadata(expected_cache_meta)
            print(f"Parquet cache saved → {CLEANED_PARQUET_PATH.resolve()}")
        return cached_df

    prepared_df = source_df.copy()
    prepared_df["cleaned_text"] = preprocess_texts(
        prepared_df["review"].fillna("").tolist(),
        batch_size=PREPROCESS_BATCH_SIZE,
        n_process=PREPROCESS_N_PROCESS,
    )
    prepared_df.to_parquet(CLEANED_PARQUET_PATH, index=False)
    write_cache_metadata(expected_cache_meta)
    print(f"Cleaned data saved → {CLEANED_PARQUET_PATH.resolve()}")
    return prepared_df


df = load_or_preprocess_reviews(df)

print("\nПримеры предобработки:")
for _, row in df.sample(3, random_state=SEED).iterrows():
    print("ORIGINAL:", str(row["review"])[:120])
    print("CLEANED: ", str(row["cleaned_text"])[:120])
    print()


Found cleaned cache: D:\DEV\ML2\LAB_IV\data\dataset\kinopoisk_reviews_cleaned.parquet
Loaded cleaned data from cache → D:\DEV\ML2\LAB_IV\data\dataset\kinopoisk_reviews_cleaned.parquet

Примеры предобработки:
ORIGINAL: Фильм-биография рок-н-рольной группы The Runaways, снятый режиссёром Флорией Сигизмонди (Postmortem Bliss, Lest We Forge
CLEANED:  фильм биография рок рольный группа the runaways снять режиссёр флорией сигизмонди postmortem bliss lest we forget главны

ORIGINAL: Хотели бы вы снова окунуться в омут ночной Москвы, каждодневной роскоши и бесконечной вечеринки? Феликс Михайлов и его к
CLEANED:  окунуться омут ночной москва каждодневный роскошь бесконечный вечеринка феликс михайл команда пораскинуть мозг решить до

ORIGINAL: Речь пойдет исключительно про первый сезон (т. к. на этом я сказал себе 'стоп' и дальше не продолжал).

Вообще, это дово
CLEANED:  речь пойти исключительно первый сезон сказать стоп далёкий не продолжать противный чувство потратить немало время просмо



### 4.4 Проверка качества предобработки

Проверяем, что предобработка не породила пустых строк,
и смотрим на по одному примеру каждого класса.


In [26]:
empty_mask = df["cleaned_text"].str.strip().str.len() == 0
print(f"Пустых cleaned_text: {empty_mask.sum()}")
df = df[~empty_mask].reset_index(drop=True)

for cls in df["sentiment"].unique():
    print(f"\n--- {cls.upper()} ---")
    row = df[df["sentiment"] == cls].sample(1, random_state=SEED).iloc[0]
    print("ORIGINAL:", str(row["review"])[:150])
    print("CLEANED: ", str(row["cleaned_text"])[:150])


Пустых cleaned_text: 0

--- NEGATIVE ---
ORIGINAL: Первый вопрос, который посещает после просмотра фильма: зачем ориентироваться на взрослую аудиторию? (Рейтинг 'R', напоминаю) Неужели нельзя было убра
CLEANED:  первый вопрос посещать просмотр фильм ориентироваться взрослый аудитория рейтинг напоминать убрать нецензурный брань сделать видеоряд мягкий юмор посы

--- NEUTRAL ---
ORIGINAL: Гигантский ящер Годзилла давно стал одним из самых популярных персонажей массовой культуры. Японские фильмы про огромного монстра завоевали широкую по
CLEANED:  гигантский ящер годзилла популярный персонаж массовый культура японский фильм огромный монстр завоевать широкий популярность считаться классика мирово

--- POSITIVE ---
ORIGINAL: Есть такой блестящий писатель Джон Ридли. Он выработал такой стиль повествования, что в любой книге по итогу всех лихих передряг не сочувствуешь ни од
CLEANED:  блестящий писатель джон ридли выработать стиль повествование книга итог лихой передряга не сочувствуешь ни мн

> **Вывод:**

На полном датасете пустых `cleaned_text` получилось `0`, то есть предобработка не уничтожила ни один отзыв полностью. По примерам видно, что тексты остаются читаемыми: сохраняются ключевые существительные и прилагательные вроде `режиссёр`, `аудитория`, `популярный`, `классика`, `блестящий`, а также отрицания типа `не`.

Признаков слишком агрессивной очистки я не вижу. Да, часть форм выглядит немного шероховато из-за spaCy и в тексте остаются английские названия вроде `the runaways`, но для TF-IDF это скорее допустимый шум. Стоп-слова, регулярку и параметры `nlp.pipe` в этой версии можно оставить без изменений.


## Подготовка данных к обучению

Перед обучением моделей выполняем два обязательных шага:
стратифицированное разделение на обучающую и тестовую выборки,
а затем TF-IDF векторизацию очищенных текстов.
Базовые модели используют общий TF-IDF для честного нулевого сравнения,
а Optuna-модели получают собственный `TfidfVectorizer` внутри `Pipeline`,
чтобы словарь и IDF обучались заново на каждом CV-фолде.


### 5.1 Train / test split

Стратифицированное разделение 80 / 20 с фиксированным `SEED`
обеспечивает одинаковое соотношение классов в обеих частях.


In [27]:
X = df["cleaned_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {len(X_train)},  Test: {len(X_test)}")
print("\nТренировочное распределение классов:")
print(y_train.value_counts(normalize=True).round(3))


Train: 105335,  Test: 26334

Тренировочное распределение классов:
sentiment
positive    0.662
neutral     0.188
negative    0.151
Name: proportion, dtype: float64


### 5.2 TF-IDF векторизация

`TfidfVectorizer` с параметрами, рекомендованными для задач классификации текста:
- `sublinear_tf=True` — логарифмическое масштабирование TF снижает влияние очень частых слов;
- `ngram_range=(1, 2)` — учёт биграмм захватывает контекст «не хороший»;
- `max_features=50 000` — ограничение размерности для управляемости;
- `min_df=2` — игнорируем слова, встречающиеся только один раз (шум).


In [28]:
TFIDF_BASE_PARAMS = {
    "sublinear_tf": True,
    "ngram_range": (1, 2),
    "max_features": 50_000,
    "min_df": 2,
    "analyzer": "word",
}

tfidf = TfidfVectorizer(**TFIDF_BASE_PARAMS)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("Train matrix:", X_train_tfidf.shape)
print("Test  matrix:", X_test_tfidf.shape)
print(f"Словарь TF-IDF: {len(tfidf.vocabulary_):,} признаков")

Train matrix: (105335, 50000)
Test  matrix: (26334, 50000)
Словарь TF-IDF: 50,000 признаков


> **Вывод:**

После стратифицированного разделения получилось `105335` train-объектов и `26334` test-объекта, а TF-IDF дал матрицы `(105335, 50000)` и `(26334, 50000)`. Лимит словаря в `50,000` признаков полностью выбран, значит текстового разнообразия в датасете хватает с большим запасом.

Для baseline-моделей такая размерность нормальна: матрица разреженная, а линейные модели как раз хорошо работают с TF-IDF. В Optuna векторизатор правильно оставлен внутри `Pipeline`, потому что иначе словарь и IDF могли бы подглядывать в валидационные фолды, а это уже не совсем честная оценка.


## Классификация

Основной раздел работы. Сравниваем три обязательные baseline-модели:
LogisticRegression, LinearSVC и CatBoostClassifier.
Затем для каждой из этих трёх моделей подбираем гиперпараметры через Optuna,
добавляем TF-IDF + LSA pipeline как отдельную конфигурацию
и выбираем лучшую модель по macro F1.

Режим обучения CatBoost берётся из `DEVICE_MODE`: на CUDA-машине можно включить
GPU, а на macOS или другом окружении без CUDA ноутбук остаётся на CPU.


### 6.1 Базовые модели (нулевой проход)

Сначала задаём три стартовые модели на общей TF-IDF матрице, затем отдельной ячейкой обучаем их и считаем метрики. CatBoost берёт `CATBOOST_DEVICE_PARAMS`, поэтому код одинаково работает на CPU и CUDA.


In [29]:
import time

catboost_loss = "MultiClass" if y_train.nunique() > 2 else "Logloss"

MODELS_BASELINE = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=SEED
    ),
    "LinearSVC": LinearSVC(
        max_iter=3000, class_weight="balanced", random_state=SEED
    ),
    "CatBoostClassifier": CatBoostClassifier(
        **CATBOOST_DEVICE_PARAMS,
        iterations=500,
        learning_rate=0.1,
        depth=6,
        loss_function=catboost_loss,
        auto_class_weights="Balanced",
        random_seed=SEED,
        verbose=0,
        thread_count=-1,
        allow_writing_files=False,
    ),
}


### 6.1.1 Обучение baseline-моделей

Одинаково обучаем каждую модель, считаем `macro_f1`, `accuracy` и время. Это нулевая точка перед Optuna.


In [30]:
def evaluate_baseline_model(name: str, model) -> dict:
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    preds = np.array(model.predict(X_test_tfidf)).flatten()
    elapsed = time.time() - t0
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)
    print(f"{name:<25} macro F1={f1:.4f}  acc={acc:.4f}  ({elapsed:.2f}s)")
    return {"macro_f1": round(f1, 4), "accuracy": round(acc, 4), "time_s": round(elapsed, 2)}


baseline_results = {
    name: evaluate_baseline_model(name, model)
    for name, model in MODELS_BASELINE.items()
}
baseline_df = pd.DataFrame(baseline_results).T.sort_values("macro_f1", ascending=False)
display(baseline_df)


LogisticRegression        macro F1=0.6519  acc=0.7323  (9.75s)
LinearSVC                 macro F1=0.6343  acc=0.7433  (10.30s)
CatBoostClassifier        macro F1=0.6134  acc=0.6933  (135.20s)


,macro_f1,accuracy,time_s
LogisticRegression,0.6519,0.7323,9.75
LinearSVC,0.6343,0.7433,10.30
CatBoostClassifier,0.6134,0.6933,135.20


> **Вывод:**

Лучшей baseline-моделью по `macro_f1` стала `LogisticRegression`: `macro_f1=0.6519`, `accuracy=0.7323`. `LinearSVC` дал более высокую `accuracy=0.7433`, но по `macro_f1` просел до `0.6343`, то есть хуже балансирует качество по классам. По времени `LogisticRegression` и `LinearSVC` почти одинаковые (`9.39s` и `9.89s`), а `CatBoostClassifier` оказался заметно тяжелее — `119.60s`.

CatBoost в нулевом проходе выглядит слабее линейных моделей: `macro_f1=0.6134`, `accuracy=0.6933`. В целом это ожидаемо для sparse TF-IDF: простые линейные границы часто оказываются сильнее и быстрее, чем бустинг, которому такая высокая разреженная размерность дается не очень естественно.


### 6.2 Optuna — пространства поиска

Optuna подбирает параметры TF-IDF и классификатора по `f1_macro` на стратифицированной CV. Ниже отдельно вынесен пресет CatBoost (`easy`, `medium`, `hard`, `extreme`), а затем небольшими ячейками описаны общие настройки, пространства моделей, кэш и единая функция запуска.


In [31]:
# Выберите один из пресетов: "easy", "medium", "hard" или "extreme".
# extreme запускает CatBoost Optuna на всем train, как линейные модели.
CATBOOST_OPTUNA_PRESET = "medium"

CATBOOST_OPTUNA_PRESETS = {
    "easy": {
        "description": "быстрый smoke-подбор для проверки пайплайна",
        "n_trials": 6,
        "timeout": 15 * 60,
        "max_samples": 8_000,
        "cv_splits": 2,
        "tfidf_max_features": (5_000, 10_000),
        "iterations": (150, 350, 50),
        "depth": (4, 6),
        "cache_version": "v4-catboost-easy",
    },
    "medium": {
        "description": "сбалансированный профиль по скорости и качеству",
        "n_trials": 10,
        "timeout": 25 * 60,
        "max_samples": 16_000,
        "cv_splits": 2,
        "tfidf_max_features": (5_000, 10_000, 15_000),
        "iterations": (200, 500, 50),
        "depth": (4, 7),
        "cache_version": "v3-balanced-catboost",
    },
    "hard": {
        "description": "более широкий подбор для качества без возврата к полному перебору",
        "n_trials": 16,
        "timeout": 40 * 60,
        "max_samples": 50_000,
        "cv_splits": 3,
        "tfidf_max_features": (10_000, 15_000, 20_000, 30_000),
        "iterations": (300, 700, 50),
        "depth": (5, 8),
        "cache_version": "v4-catboost-hard",
    },
    "extreme": {
        "description": "полный CV-подбор CatBoost на всем train без стратифицированного сэмпла",
        "n_trials": 20,
        "timeout": 120 * 60,
        "max_samples": None,
        "cv_splits": 3,
        "tfidf_max_features": (10_000, 15_000, 20_000, 30_000),
        "iterations": (300, 700, 50),
        "depth": (5, 8),
        "cache_version": "v5-catboost-extreme",
    },
}

if CATBOOST_OPTUNA_PRESET not in CATBOOST_OPTUNA_PRESETS:
    available_presets = ", ".join(CATBOOST_OPTUNA_PRESETS)
    raise ValueError(f"Неизвестный CatBoost Optuna preset: {CATBOOST_OPTUNA_PRESET}. Доступно: {available_presets}")

CATBOOST_OPTUNA_CONFIG = CATBOOST_OPTUNA_PRESETS[CATBOOST_OPTUNA_PRESET]
print(f"CatBoost Optuna preset: {CATBOOST_OPTUNA_PRESET} — {CATBOOST_OPTUNA_CONFIG['description']}")


CatBoost Optuna preset: medium — сбалансированный профиль по скорости и качеству


### 6.2.1 Общие настройки Optuna

Фиксируем число trials, timeout-ы и директорию кэша. Для CatBoost значения подтягиваются из выбранного пресета.


In [32]:
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

N_OPTUNA_TRIALS = 20
OPTUNA_TRIALS_BY_MODEL = {
    "LogisticRegression": N_OPTUNA_TRIALS,
    "LinearSVC": N_OPTUNA_TRIALS,
    "CatBoostClassifier": CATBOOST_OPTUNA_CONFIG["n_trials"],
}
OPTUNA_TIMEOUT_BY_MODEL = {
    "LogisticRegression": None,
    "LinearSVC": None,
    "CatBoostClassifier": CATBOOST_OPTUNA_CONFIG["timeout"],
}

OPTUNA_CACHE_VERSION = "v3-balanced-catboost"
OPTUNA_CACHE_DIR = DATA_PATH.with_name("optuna_cache")
OPTUNA_CACHE_DIR.mkdir(parents=True, exist_ok=True)


### 6.2.2 Пространство TF-IDF

Эти параметры общие для всех tuned-моделей: размер словаря, `min_df`, биграммы и `sublinear_tf`.


In [33]:
def make_tfidf(params: dict) -> TfidfVectorizer:
    return TfidfVectorizer(
        analyzer="word",
        min_df=params["tfidf_min_df"],
        max_features=params["tfidf_max_features"],
        ngram_range=(1, params["tfidf_ngram_max"]),
        sublinear_tf=params["tfidf_sublinear_tf"],
    )


def suggest_tfidf_params(trial, *, max_features_choices=(30_000, 50_000, 80_000)) -> dict:
    return {
        "tfidf_min_df": trial.suggest_int("tfidf_min_df", 1, 5),
        "tfidf_max_features": trial.suggest_categorical("tfidf_max_features", list(max_features_choices)),
        "tfidf_ngram_max": trial.suggest_categorical("tfidf_ngram_max", [1, 2]),
        "tfidf_sublinear_tf": trial.suggest_categorical("tfidf_sublinear_tf", [True, False]),
    }


### 6.2.3 Пространства моделей

Для каждой модели задаём свой классификатор, но оставляем одинаковую схему `Pipeline`: сначала TF-IDF, потом модель.


In [34]:
def suggest_lr_params(trial) -> dict:
    params = suggest_tfidf_params(trial)
    params.update({"clf_C": trial.suggest_float("clf_C", 0.05, 10.0, log=True)})
    return params


def build_lr_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", LogisticRegression(C=params["clf_C"], max_iter=1000, class_weight="balanced", random_state=SEED)),
    ])


def suggest_svc_params(trial) -> dict:
    params = suggest_tfidf_params(trial)
    params.update({
        "clf_C": trial.suggest_float("clf_C", 0.05, 10.0, log=True),
        "clf_tol": trial.suggest_float("clf_tol", 1e-5, 1e-3, log=True),
    })
    return params


def build_svc_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", LinearSVC(
            C=params["clf_C"],
            tol=params["clf_tol"],
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED,
        )),
    ])


def suggest_catboost_params(trial) -> dict:
    iter_low, iter_high, iter_step = CATBOOST_OPTUNA_CONFIG["iterations"]
    depth_low, depth_high = CATBOOST_OPTUNA_CONFIG["depth"]
    params = suggest_tfidf_params(trial, max_features_choices=CATBOOST_OPTUNA_CONFIG["tfidf_max_features"])
    params.update({
        "clf_iterations": trial.suggest_int("clf_iterations", iter_low, iter_high, step=iter_step),
        "clf_depth": trial.suggest_int("clf_depth", depth_low, depth_high),
        "clf_learning_rate": trial.suggest_float("clf_learning_rate", 0.05, 0.3, log=True),
        "clf_l2_leaf_reg": trial.suggest_float("clf_l2_leaf_reg", 1.0, 10.0, log=True),
    })
    return params


def build_catboost_pipeline(params: dict) -> Pipeline:
    return Pipeline([
        ("tfidf", make_tfidf(params)),
        ("clf", CatBoostClassifier(
            **CATBOOST_DEVICE_PARAMS,
            iterations=params["clf_iterations"],
            depth=params["clf_depth"],
            learning_rate=params["clf_learning_rate"],
            l2_leaf_reg=params["clf_l2_leaf_reg"],
            loss_function=catboost_loss,
            auto_class_weights="Balanced",
            random_seed=SEED,
            verbose=0,
            thread_count=-1,
            allow_writing_files=False,
        )),
    ])


### 6.2.4 Сэмпл CatBoost и кэш моделей

Здесь собраны служебные функции для стратифицированного сэмпла, SQLite-study и joblib-кэша финальных tuned-моделей.


In [35]:
def make_stratified_optuna_sample(X_values, y_values, max_samples: int | None):
    if max_samples is None or len(X_values) <= max_samples:
        return X_values, y_values

    X_sample, _, y_sample, _ = train_test_split(
        X_values,
        y_values,
        train_size=max_samples,
        random_state=SEED,
        stratify=y_values,
    )
    return X_sample, y_sample


def safe_cache_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


def model_optuna_cache_version(name: str) -> str:
    if name == "CatBoostClassifier":
        return CATBOOST_OPTUNA_CONFIG["cache_version"]
    return OPTUNA_CACHE_VERSION


def optuna_storage_path(cache_version: str) -> Path:
    return OPTUNA_CACHE_DIR / f"studies_{cache_version}.sqlite3"


def optuna_storage_url(cache_version: str) -> str:
    return f"sqlite:///{optuna_storage_path(cache_version).as_posix()}"


def json_safe(value):
    return json.loads(json.dumps(value, ensure_ascii=False))


def completed_trials_count(study) -> int:
    return sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials)


def optuna_model_cache_paths(name: str) -> tuple[Path, Path]:
    cache_version = model_optuna_cache_version(name)
    stem = f"{cache_version}_{safe_cache_name(name)}"
    return OPTUNA_CACHE_DIR / f"{stem}.joblib", OPTUNA_CACHE_DIR / f"{stem}.meta.json"


def build_optuna_model_metadata(name: str, best_params: dict, X_fit, y_fit) -> dict:
    return {
        "cache_version": model_optuna_cache_version(name),
        "model": name,
        "seed": SEED,
        "best_params": best_params,
        "train_rows": int(len(X_fit)),
        "test_rows": int(len(X_test)),
        "target_counts": {str(label): int(count) for label, count in y_fit.value_counts().sort_index().items()},
        "catboost_device_params": CATBOOST_DEVICE_PARAMS if name == "CatBoostClassifier" else None,
        "catboost_loss": catboost_loss if name == "CatBoostClassifier" else None,
        "catboost_optuna_preset": CATBOOST_OPTUNA_PRESET if name == "CatBoostClassifier" else None,
        "catboost_optuna_config": json_safe(CATBOOST_OPTUNA_CONFIG) if name == "CatBoostClassifier" else None,
    }


def load_cached_optuna_model(name: str, expected_meta: dict):
    model_path, meta_path = optuna_model_cache_paths(name)
    if not model_path.exists() or not meta_path.exists():
        return None

    try:
        with meta_path.open("r", encoding="utf-8") as meta_file:
            actual_meta = json.load(meta_file)
    except (OSError, json.JSONDecodeError):
        return None

    if actual_meta != expected_meta:
        return None

    try:
        return load(model_path)
    except Exception as exc:
        print(f"Не удалось прочитать кэш модели {name}: {exc}. Модель будет обучена заново.")
        return None


def save_cached_optuna_model(name: str, model, meta: dict) -> Path:
    model_path, meta_path = optuna_model_cache_paths(name)
    dump(model, model_path)
    with meta_path.open("w", encoding="utf-8") as meta_file:
        json.dump(meta, meta_file, ensure_ascii=False, indent=2)
    return model_path


### 6.2.5 Единый запуск Optuna

Функция запускает study, донабирает недостающие trials, обучает лучшую модель на полном train и возвращает все метрики для таблицы.


In [36]:
def optimize_with_optuna(
    name: str,
    suggest_params,
    build_pipeline,
    *,
    cv_n_jobs: int = -1,
    cv=CV,
    X_opt=None,
    y_opt=None,
    n_trials: int | None = None,
    timeout: int | None = None,
) -> dict:
    X_search = X_train if X_opt is None else X_opt
    y_search = y_train if y_opt is None else y_opt
    target_trials = n_trials if n_trials is not None else OPTUNA_TRIALS_BY_MODEL.get(name, N_OPTUNA_TRIALS)
    timeout_seconds = OPTUNA_TIMEOUT_BY_MODEL.get(name) if timeout is None else timeout
    cache_version = model_optuna_cache_version(name)

    def objective(trial):
        params = suggest_params(trial)
        pipeline = build_pipeline(params)
        scores = cross_val_score(
            pipeline,
            X_search,
            y_search,
            cv=cv,
            scoring="f1_macro",
            n_jobs=cv_n_jobs,
        )
        trial.set_user_attr("cv_std", float(scores.std()))
        return float(scores.mean())

    study = optuna.create_study(
        study_name=f"{cache_version}_{safe_cache_name(name)}",
        storage=optuna_storage_url(cache_version),
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    completed_before_start = completed_trials_count(study)
    remaining_trials = max(0, target_trials - completed_before_start)
    started_at = time.time()
    with tqdm(
        total=target_trials,
        initial=min(completed_before_start, target_trials),
        desc=f"Optuna: {name}",
        dynamic_ncols=True,
        mininterval=1.0,
        leave=False,
    ) as progress:
        for _ in range(remaining_trials):
            remaining_time = None
            if timeout_seconds is not None:
                remaining_time = max(0.0, timeout_seconds - (time.time() - started_at))
                if remaining_time == 0.0:
                    break

            completed_before = len(study.trials)
            study.optimize(objective, n_trials=1, timeout=remaining_time, show_progress_bar=False)
            completed_delta = len(study.trials) - completed_before
            if completed_delta == 0:
                break

            progress.update(completed_delta)
            try:
                progress.set_postfix(best=f"{study.best_value:.4f}")
            except ValueError:
                pass

    completed_trials = completed_trials_count(study)
    if completed_trials == 0:
        raise RuntimeError(f"Optuna для {name} не завершила ни одного trial.")

    best_params = study.best_params
    model_meta = build_optuna_model_metadata(name, best_params, X_train, y_train)
    best_model = load_cached_optuna_model(name, model_meta)
    fit_from_cache = best_model is not None

    if fit_from_cache:
        elapsed = 0.0
        print(f"{name}: загружена обученная модель из кэша.")
    else:
        best_model = build_pipeline(best_params)
        print(f"{name}: финальное обучение на {len(X_train):,} строках...")
        t0 = time.time()
        best_model.fit(X_train, y_train)
        elapsed = time.time() - t0
        model_path = save_cached_optuna_model(name, best_model, model_meta)
        print(f"{name}: модель сохранена в кэш → {model_path.resolve()}")

    preds = np.array(best_model.predict(X_test)).flatten()
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)

    print(f"\n{name}")
    print("Best CV macro F1:", round(study.best_value, 4))
    print(f"Completed trials: {completed_trials}/{target_trials}; search rows: {len(X_search):,}")
    print("Best params:", best_params)
    print(f"Test macro F1={f1:.4f}  acc={acc:.4f}  fit_time={elapsed:.2f}s  cached={fit_from_cache}")
    print("\n", classification_report(y_test, preds))

    return {
        "study": study,
        "model": best_model,
        "preds": preds,
        "macro_f1": f1,
        "accuracy": acc,
        "fit_time_s": elapsed,
        "fit_from_cache": fit_from_cache,
        "completed_trials": completed_trials,
        "target_trials": target_trials,
        "search_rows": len(X_search),
    }


### 6.3 Optuna — запуск оптимизации трёх моделей

Запуск разделён на три ячейки: подготовка CatBoost-сэмпла, сами Optuna-прогоны и сборка итоговой таблицы. Так проще увидеть, где именно тратится время и какие результаты дальше используются.


In [37]:
catboost_cv = StratifiedKFold(
    n_splits=CATBOOST_OPTUNA_CONFIG["cv_splits"],
    shuffle=True,
    random_state=SEED,
)
catboost_X_opt, catboost_y_opt = make_stratified_optuna_sample(
    X_train,
    y_train,
    CATBOOST_OPTUNA_CONFIG["max_samples"],
)
print(
    f"CatBoost Optuna preset: {CATBOOST_OPTUNA_PRESET}; "
    f"trials={CATBOOST_OPTUNA_CONFIG['n_trials']}; "
    f"cv={CATBOOST_OPTUNA_CONFIG['cv_splits']}-fold"
)
print(f"CatBoost Optuna search rows: {len(catboost_X_opt):,} / {len(X_train):,}")


CatBoost Optuna preset: medium; trials=10; cv=2-fold
CatBoost Optuna search rows: 16,000 / 105,335


### 6.3.1 Запуск Optuna

Линейные модели подбираются на полном train, а CatBoost использует настройки выбранного пресета: сэмпл или полный train, число folds, trials и timeout.


In [38]:
lr_run = optimize_with_optuna(
    "LogisticRegression",
    suggest_lr_params,
    build_lr_pipeline,
    cv_n_jobs=-1,
)
svc_run = optimize_with_optuna(
    "LinearSVC",
    suggest_svc_params,
    build_svc_pipeline,
    cv_n_jobs=-1,
)
catboost_run = optimize_with_optuna(
    "CatBoostClassifier",
    suggest_catboost_params,
    build_catboost_pipeline,
    cv_n_jobs=1,
    cv=catboost_cv,
    X_opt=catboost_X_opt,
    y_opt=catboost_y_opt,
    n_trials=CATBOOST_OPTUNA_CONFIG["n_trials"],
    timeout=CATBOOST_OPTUNA_CONFIG["timeout"],
)


LogisticRegression: загружена обученная модель из кэша.

LogisticRegression
Best CV macro F1: 0.6452
Completed trials: 20/20; search rows: 105,335
Best params: {'tfidf_min_df': 4, 'tfidf_max_features': 80000, 'tfidf_ngram_max': 2, 'tfidf_sublinear_tf': True, 'clf_C': 1.9169322117061403}
Test macro F1=0.6492  acc=0.7356  fit_time=0.00s  cached=True

               precision    recall  f1-score   support

    negative       0.63      0.71      0.67      3965
     neutral       0.40      0.46      0.43      4941
    positive       0.89      0.82      0.85     17428

    accuracy                           0.74     26334
   macro avg       0.64      0.66      0.65     26334
weighted avg       0.76      0.74      0.74     26334



LinearSVC: загружена обученная модель из кэша.

LinearSVC
Best CV macro F1: 0.6444
Completed trials: 20/20; search rows: 105,335
Best params: {'tfidf_min_df': 4, 'tfidf_max_features': 80000, 'tfidf_ngram_max': 2, 'tfidf_sublinear_tf': True, 'clf_C': 0.13871518308271139, 'clf_tol': 0.0002769950817401335}
Test macro F1=0.6518  acc=0.7714  fit_time=0.00s  cached=True

               precision    recall  f1-score   support

    negative       0.65      0.74      0.69      3965
     neutral       0.48      0.32      0.39      4941
    positive       0.85      0.91      0.88     17428

    accuracy                           0.77     26334
   macro avg       0.66      0.66      0.65     26334
weighted avg       0.75      0.77      0.76     26334



CatBoostClassifier: загружена обученная модель из кэша.



CatBoostClassifier
Best CV macro F1: 0.5636
Completed trials: 10/10; search rows: 16,000
Best params: {'tfidf_min_df': 2, 'tfidf_max_features': 10000, 'tfidf_ngram_max': 1, 'tfidf_sublinear_tf': False, 'clf_iterations': 450, 'clf_depth': 6, 'clf_learning_rate': 0.19902108036086422, 'clf_l2_leaf_reg': 3.11742200300463}
Test macro F1=0.6133  acc=0.6927  fit_time=0.00s  cached=True

               precision    recall  f1-score   support

    negative       0.55      0.70      0.62      3965
     neutral       0.36      0.47      0.41      4941
    positive       0.88      0.75      0.81     17428

    accuracy                           0.69     26334
   macro avg       0.60      0.64      0.61     26334
weighted avg       0.74      0.69      0.71     26334



### 6.3.2 Сводка tuned-моделей

Из каждого Optuna-прогона достаём лучшую модель, предсказания и метрики, затем собираем компактную таблицу сравнения.


In [39]:
def unpack_optuna_run(run: dict):
    return run["model"], run["preds"], run["macro_f1"], run["accuracy"]


best_lr, lr_preds, lr_f1, lr_acc = unpack_optuna_run(lr_run)
best_svc, svc_preds, svc_f1, svc_acc = unpack_optuna_run(svc_run)
best_catboost, catboost_optuna_preds, catboost_f1, catboost_acc = unpack_optuna_run(catboost_run)

optuna_summary_df = pd.DataFrame({
    "LR tuned": {
        "cv_macro_f1": lr_run["study"].best_value,
        "test_macro_f1": lr_f1,
        "test_accuracy": lr_acc,
        "trials": lr_run["completed_trials"],
        "search_rows": lr_run["search_rows"],
    },
    "LinearSVC tuned": {
        "cv_macro_f1": svc_run["study"].best_value,
        "test_macro_f1": svc_f1,
        "test_accuracy": svc_acc,
        "trials": svc_run["completed_trials"],
        "search_rows": svc_run["search_rows"],
    },
    "CatBoost tuned": {
        "cv_macro_f1": catboost_run["study"].best_value,
        "test_macro_f1": catboost_f1,
        "test_accuracy": catboost_acc,
        "trials": catboost_run["completed_trials"],
        "search_rows": catboost_run["search_rows"],
    },
}).T.sort_values("test_macro_f1", ascending=False)

display(optuna_summary_df.round(4))


,cv_macro_f1,test_macro_f1,test_accuracy,trials,search_rows
LinearSVC tuned,0.6444,0.6518,0.7714,20.0,105335.0
LR tuned,0.6452,0.6492,0.7356,20.0,105335.0
CatBoost tuned,0.5636,0.6133,0.6927,10.0,16000.0


> **Вывод:**

Среди tuned-моделей лучше всех получился `LinearSVC tuned`: `test_macro_f1=0.6518`, `accuracy=0.7714`. Для `LogisticRegression` Optuna фактически не дала прироста по `macro_f1`: было `0.6519`, стало `0.6492`, зато `accuracy` чуть выросла до `0.7356`. У `LinearSVC` прирост уже заметнее: `macro_f1` вырос с `0.6343` до `0.6518`, а `accuracy` с `0.7433` до `0.7714`. CatBoost в последнем `hard`-подборе поднялся с `0.6134` до `0.6271`, но все равно не догнал линейные модели.

Самые важные параметры для отчета: у линейных моделей Optuna выбрала `tfidf_min_df=4`, `tfidf_max_features=80000`, `ngram_max=2` и `sublinear_tf=True`; для `LinearSVC` удачным оказался небольшой `C=0.1387`, а для `LogisticRegression` — `C=1.9169`. Разрыв между CV и test небольшой для `LR` и `LinearSVC` (`0.6452 → 0.6492`, `0.6444 → 0.6518`), а у CatBoost test выше CV (`0.6043 → 0.6271`), вероятно из-за того, что подбор шел только на `50,000` train-строках.


### 6.4 TF-IDF + LSA (TruncatedSVD)

Латентно-семантический анализ (LSA) сжимает TF-IDF матрицу до 300 скрытых тем,
захватывая синонимию и тематическую близость слов.
Сравниваем с прямым TF-IDF подходом как отдельную дополнительную конфигурацию.


In [40]:
lsa_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(**TFIDF_BASE_PARAMS)),
    ("svd",   TruncatedSVD(n_components=300, random_state=SEED)),
    ("clf",   LogisticRegression(
        C=5.0, max_iter=1000, class_weight="balanced", random_state=SEED
    )),
])

lsa_pipeline.fit(X_train, y_train)
lsa_preds = lsa_pipeline.predict(X_test)
lsa_f1  = f1_score(y_test, lsa_preds, average="macro")
lsa_acc = accuracy_score(y_test, lsa_preds)
print(f"LSA Pipeline  macro F1={lsa_f1:.4f}  acc={lsa_acc:.4f}")
print("\n", classification_report(y_test, lsa_preds))

LSA Pipeline  macro F1=0.6187  acc=0.6903

               precision    recall  f1-score   support

    negative       0.57      0.73      0.64      3965
     neutral       0.35      0.49      0.41      4941
    positive       0.89      0.74      0.81     17428

    accuracy                           0.69     26334
   macro avg       0.60      0.65      0.62     26334
weighted avg       0.74      0.69      0.71     26334



> **Вывод:**

`TF-IDF + LSA` дал `macro_f1=0.6187` и `accuracy=0.6903`, то есть сжатие до `300` компонент ухудшило результат относительно прямого TF-IDF. Особенно заметна просадка на `neutral`: `precision=0.35`, `f1-score=0.41`. У `positive` точность высокая (`0.89`), но `recall=0.74`, из-за чего модель теряет много положительных отзывов.

В моем случае LSA лучше оставить как сравнительный эксперимент, а не как рабочий вариант. Похоже, сжатие убрало часть редких, но полезных маркеров тональности, и для этой задачи линейная модель на прямом TF-IDF оказывается более практичной.


### 6.5 Сводная таблица результатов

Сравниваем baseline, Optuna-tuned модели и TF-IDF + LSA по accuracy и macro F1
на тестовой выборке. Лучшая модель дальше используется для анализа ошибок
и проверки на новых текстах.


In [41]:
def get_scores(model, X, y_true):
    preds = np.array(model.predict(X)).flatten()
    return {
        "macro_f1": round(f1_score(y_true, preds, average="macro"), 4),
        "accuracy": round(accuracy_score(y_true, preds), 4),
    }


results_df = pd.DataFrame.from_dict({
    "LR baseline": get_scores(MODELS_BASELINE["LogisticRegression"], X_test_tfidf, y_test),
    "LinearSVC baseline": get_scores(MODELS_BASELINE["LinearSVC"], X_test_tfidf, y_test),
    "CatBoost baseline": get_scores(MODELS_BASELINE["CatBoostClassifier"], X_test_tfidf, y_test),
    "LR tuned": {"macro_f1": lr_f1, "accuracy": lr_acc},
    "LinearSVC tuned": {"macro_f1": svc_f1, "accuracy": svc_acc},
    "CatBoost tuned": {"macro_f1": catboost_f1, "accuracy": catboost_acc},
    "TF-IDF + LSA": {"macro_f1": lsa_f1, "accuracy": lsa_acc},
}, orient="index")
results_df = results_df.astype(float).sort_values("macro_f1", ascending=False)

display(results_df)

best_model_name = results_df.index[0]
print(f"\nЛучшая модель: {best_model_name}")


,macro_f1,accuracy
LR baseline,0.651900,0.732300
LinearSVC tuned,0.651774,0.771398
LR tuned,0.649208,0.735551
LinearSVC baseline,0.634300,0.743300
TF-IDF + LSA,0.618723,0.690324
CatBoost baseline,0.613400,0.693300
CatBoost tuned,0.613309,0.692717



Лучшая модель: LR baseline


> **Вывод:**

По сводной таблице итоговой лучшей моделью по `macro_f1` остается `LR baseline`: `macro_f1=0.6519`, `accuracy=0.7323`. Лучшее значение `accuracy` при этом у `LinearSVC tuned` — `0.7714`, но его `macro_f1=0.651774` буквально на `0.000126` ниже, чем у `LR baseline`. То есть универсального победителя тут нет: одна модель чуть лучше по балансу классов, другая заметно лучше по общей доле правильных ответов.

Если выбирать самый разумный вариант по качеству, скорости и простоте запуска, я бы оставил `LR baseline`. Она обучается быстро, не требует Optuna и дает лучший `macro_f1`. `LinearSVC tuned` тоже выглядит сильным кандидатом, но его преимущество скорее в `accuracy`, а не в главной для этой задачи метрике.


### 6.6 Матрица ошибок лучшей модели

Матрица ошибок показывает абсолютные числа правильных и неправильных
классификаций для каждой пары (истинный класс, предсказанный класс).

In [42]:
preds_map = {
    "LR baseline": MODELS_BASELINE["LogisticRegression"].predict(X_test_tfidf),
    "LinearSVC baseline": MODELS_BASELINE["LinearSVC"].predict(X_test_tfidf),
    "CatBoost baseline": np.array(MODELS_BASELINE["CatBoostClassifier"].predict(X_test_tfidf)).flatten(),
    "LR tuned": lr_preds,
    "LinearSVC tuned": svc_preds,
    "CatBoost tuned": catboost_optuna_preds,
    "TF-IDF + LSA": lsa_preds,
}

best_preds = preds_map[best_model_name]
labels_sorted = sorted(y_test.unique())

plot_confusion_matrix_plotly(
    y_test,
    best_preds,
    labels_sorted,
    title=f"Матрица ошибок — {best_model_name}",
)


> **Вывод:**

Матрица ошибок для `LR baseline` показывает ожидаемую асимметрию. Лучше всего распознается `positive`: `13996` правильных ответов из `17428`. Хуже всего выглядит `neutral`: правильно найдено только `2402` из `4941`, а ошибки расходятся в обе стороны — `1061` раз в `negative` и `1478` раз в `positive`.

Самая частая ошибка в абсолютных числах — `positive → neutral` (`2759` случаев). Это хорошо согласуется с `macro_f1`: модель сильна на большом положительном классе, но нейтральные отзывы остаются пограничной зоной. Самые критичные ошибки для анализа тональности — прямые перепутывания `negative ↔ positive` (`213` и `673` случаев), потому что там меняется сам знак оценки, а не только степень уверенности.


## Анализ ошибок

Анализируем, какие конкретные тексты модель классифицировала неверно.
Это помогает выявить систематические слабости:
саркастические тексты, смешанные отзывы, очень короткие фрагменты.


### 7.1 Сбор ошибочных предсказаний

Собираем все тексты, где предсказание расходится с истинной меткой,
и показываем образцы ложноположительных и ложноотрицательных классификаций.

In [43]:
test_df = X_test.reset_index(drop=True).to_frame("cleaned_text")
test_df["original"]   = df.loc[X_test.index, "review"].values
test_df["true_label"] = y_test.reset_index(drop=True)
test_df["pred_label"] = best_preds

errors = test_df[test_df["true_label"] != test_df["pred_label"]].copy()
print(f"Всего ошибок: {len(errors)} из {len(test_df)} ({100*len(errors)/len(test_df):.1f}%)")
print(f"Правильно:    {len(test_df) - len(errors)} ({100*(1 - len(errors)/len(test_df)):.1f}%)")

# Show unique error combinations
print("\nРаспределение ошибок по типам:")
print(errors.groupby(["true_label", "pred_label"]).size().rename("count"))


Всего ошибок: 7050 из 26334 (26.8%)
Правильно:    19284 (73.2%)

Распределение ошибок по типам:
true_label  pred_label
negative    neutral        866
            positive       213
neutral     negative      1061
            positive      1478
positive    negative       673
            neutral       2759
Name: count, dtype: int64


### 7.1.1 Примеры ошибок

После общей таблицы ошибок выводим несколько текстов для каждого направления перепутывания классов. Это уже не метрика, а ручная проверка характера промахов.


In [44]:
# Show samples for each error type
for (true_l, pred_l), group in errors.groupby(["true_label", "pred_label"]):
    print(f"\n=== Истинный: {true_l.upper()} → Предсказан: {pred_l.upper()} ===")
    for _, row in group.head(3).iterrows():
        print(f"  [{row['true_label']}→{row['pred_label']}] {str(row['original'])[:160]}")



=== Истинный: NEGATIVE → Предсказан: NEUTRAL ===
  [negative→neutral] Хорошие фильмы в жанре мелодрамы появляются действительно редко, особенно если мелодрама носит позитивный, а не «ромеоджульеттовский» оттенок шекспировской траг
  [negative→neutral] И в очередной раз мы вынуждены наблюдать за приключениями новоявленного Индианы Джонса. Хотя в данном случае уместнее сказать Лары Крофт. Досмотрев фильм, я так
  [negative→neutral] Чего можно сказать об этом детском фильме из советского кинопрома? Да много чего. Некоторые восхищаются, культовый фильм для проживающих на 1/6 части суши (как 

=== Истинный: NEGATIVE → Предсказан: POSITIVE ===
  [negative→positive] Я не буду писать о том, какой хороший этот мультфильм. Я напишу про то 'Как испортил Disney Россия этот мультфильм'

Итак, начнём с названия, как Disney Россия 
  [negative→positive] Я хотел найти интересный фильм в жанре триллер/детектив/ужасы и наткнулся на это. После просмотра я был шокирован тем, что у фильма столь высокая оц

### 7.2 Характер ошибок

> **Вывод:**

Всего модель ошиблась на `7050` объектах из `26334`, то есть на `26.8%` тестовой выборки. Основные направления ошибок совпадают с confusion matrix: `positive → neutral` (`2759`), `neutral → positive` (`1478`), `neutral → negative` (`1061`) и `negative → neutral` (`866`). То есть чаще всего модель не радикально меняет полярность, а сдвигает отзыв в соседний или более безопасный класс.

По примерам видно, что проблемы часто связаны со смешанной оценкой, сарказмом и длинными сюжетными пересказами. Например, фрагмент `Я не буду писать о том, какой хороший этот мультфильм` содержит положительные слова, но смысл дальше негативный; а часть `positive → neutral` выглядит как почти нейтральное описание сюжета. Если говорить по простому, TF-IDF хорошо ловит отдельные маркеры, но плохо понимает разворот мысли внутри длинного текста.


## Тест на новых текстах

Проверяем лучшую модель на пяти новых отзывах в принципиально разных стилях,
пропуская их через ту же самую функцию `preprocess_text`.
Тексты написаны намеренно в разных регистрах, чтобы не подыгрывать модели.


### 8.1 Пять тестовых текстов

Пять отзывов покрывают разные стилистические регистры:
разговорный слэнг, формальный академический, сарказм, нейтральный фактический
и эмоционально насыщенный негатив.

In [45]:
test_texts = [
    # 1. Short enthusiastic slang — expected: positive
    "Топчик! Смотрел 3 раза подряд, реально крутой фильм, всем советую!",

    # 2. Formal academic-style negative — expected: negative
    "Данное кинопроизведение не отвечает минимальным требованиям качественного "
    "повествования. Режиссёрская концепция лишена внутренней логики, "
    "а актёрская игра не выдерживает никакой критики.",

    # 3. Sarcastic mixed review — expected: negative (hard case)
    "Гениальная картина, если вы любите тратить 2 часа жизни впустую. "
    "Восхитительно бессмысленный сюжет и незабываемо слабая игра актёров.",

    # 4. Very brief neutral/factual — expected: neutral if the class exists, ambiguous otherwise
    "Фильм 2019 года. Режиссёр Иванов. Посмотрел. Актёры играют нормально.",

    # 5. Emotional negative — expected: negative
    "Просто ужасно!! Деньги на ветер! Никогда не смотрите этот мусор! "
    "Хуже фильма в жизни не видел, полный провал.",
]

available_labels = set(y_train.unique())
labels_expected = [
    "positive",
    "negative",
    "negative",
    "neutral" if "neutral" in available_labels else "ambiguous",
    "negative",
]

for i, (text, expected) in enumerate(zip(test_texts, labels_expected), 1):
    print(f"[{i}] Ожидается: {expected}")
    print(f"    Текст: {text[:90]}...")
    print()

[1] Ожидается: positive
    Текст: Топчик! Смотрел 3 раза подряд, реально крутой фильм, всем советую!...

[2] Ожидается: negative
    Текст: Данное кинопроизведение не отвечает минимальным требованиям качественного повествования. Р...

[3] Ожидается: negative
    Текст: Гениальная картина, если вы любите тратить 2 часа жизни впустую. Восхитительно бессмысленн...

[4] Ожидается: neutral
    Текст: Фильм 2019 года. Режиссёр Иванов. Посмотрел. Актёры играют нормально....

[5] Ожидается: negative
    Текст: Просто ужасно!! Деньги на ветер! Никогда не смотрите этот мусор! Хуже фильма в жизни не ви...



### 8.2 Предобработка и предсказание

Пропускаем каждый текст через `preprocess_texts` —
ту же функцию, что применялась к обучающим данным.
Baseline-модели получают общий обученный `tfidf`, а tuned/LSA-модели получают
очищенный текст напрямую, потому что их векторизатор находится внутри `Pipeline`.
Для моделей с `predict_proba` показываем вероятность, для LinearSVC — margin
(`decision_function`), а не вероятность.


In [46]:
def predict_with_score(model, X):
    predictions = np.array(model.predict(X)).flatten()
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X).max(axis=1)
        score_type = "probability"
    elif hasattr(model, "decision_function"):
        raw_scores = model.decision_function(X)
        scores = np.abs(raw_scores) if raw_scores.ndim == 1 else raw_scores.max(axis=1)
        score_type = "margin"
    else:
        scores = np.full(len(predictions), np.nan)
        score_type = "n/a"
    return predictions, scores, score_type


model_obj_map = {
    "LR baseline": (MODELS_BASELINE["LogisticRegression"], "vectorized"),
    "LinearSVC baseline": (MODELS_BASELINE["LinearSVC"], "vectorized"),
    "CatBoost baseline": (MODELS_BASELINE["CatBoostClassifier"], "vectorized"),
    "LR tuned": (best_lr, "text"),
    "LinearSVC tuned": (best_svc, "text"),
    "CatBoost tuned": (best_catboost, "text"),
    "TF-IDF + LSA": (lsa_pipeline, "text"),
}

best_model_obj, input_kind = model_obj_map[best_model_name]

# Preprocess new texts with the same function used for train data
cleaned_new = preprocess_texts(test_texts, batch_size=PREPROCESS_BATCH_SIZE, n_process=PREPROCESS_N_PROCESS)

if input_kind == "text":
    predictions_new, score_values, score_type = predict_with_score(best_model_obj, cleaned_new)
else:
    new_vectors = tfidf.transform(cleaned_new)
    predictions_new, score_values, score_type = predict_with_score(best_model_obj, new_vectors)

results_new = pd.DataFrame({
    "Текст": [text[:55] + "..." for text in test_texts],
    "Ожидается": labels_expected,
    "Предсказано": predictions_new,
    "Скор модели": np.round(score_values, 3),
    "Тип скора": score_type,
})
display(results_new)


,Текст,Ожидается,Предсказано,Скор модели,Тип скора
0,"Топчик! Смотрел 3 раза подряд, реально крутой ...",positive,positive,0.626,probability
1,Данное кинопроизведение не отвечает минимальны...,negative,negative,0.802,probability
2,"Гениальная картина, если вы любите тратить 2 ч...",negative,negative,0.894,probability
3,Фильм 2019 года. Режиссёр Иванов. Посмотрел. А...,neutral,neutral,0.547,probability
4,Просто ужасно!! Деньги на ветер! Никогда не см...,negative,negative,0.926,probability


### 8.2.1 Контроль очищенных новых текстов

Показываем, во что превратились пять ручных примеров после той же предобработки, что использовалась на train.


In [47]:
print("Предобработанные тексты:")
for i, (orig, cleaned) in enumerate(zip(test_texts, cleaned_new), 1):
    print(f"[{i}] ORIGINAL: {orig[:80]}")
    print(f"    CLEANED:  {cleaned[:80]}")
    print()


Предобработанные тексты:
[1] ORIGINAL: Топчик! Смотрел 3 раза подряд, реально крутой фильм, всем советую!
    CLEANED:  топчик смотреть раз подряд реально крутой фильм советовать

[2] ORIGINAL: Данное кинопроизведение не отвечает минимальным требованиям качественного повест
    CLEANED:  кинопроизведение не отвечать минимальный требование качественный повествование р

[3] ORIGINAL: Гениальная картина, если вы любите тратить 2 часа жизни впустую. Восхитительно б
    CLEANED:  гениальный картина любить тратить час жизнь впустую восхитительно бессмысленный 

[4] ORIGINAL: Фильм 2019 года. Режиссёр Иванов. Посмотрел. Актёры играют нормально.
    CLEANED:  фильм год режиссёр иванов посмотреть актёры играть нормально

[5] ORIGINAL: Просто ужасно!! Деньги на ветер! Никогда не смотрите этот мусор! Хуже фильма в ж
    CLEANED:  ужасно деньга ветер не смотреть мусор плохой фильм жизнь не видеть полный провал



### 8.3 Комментарий к предсказаниям

> **Комментарии:**

- Текст 1, разговорный позитив: ожидалось `positive`, предсказано `positive`, скор `0.626`. Модель справилась, но не слишком уверенно: сленговое `топчик` само по себе, вероятно, не такой надежный маркер, зато помогают `крутой фильм` и `советую`.
- Текст 2, формальный негатив: ожидалось `negative`, предсказано `negative`, скор `0.802`. Здесь модель хорошо зацепилась за лексику `не отвечает`, `лишена`, `не выдерживает критики`, поэтому формальный стиль ей не помешал.
- Текст 3, сарказм: ожидалось `negative`, предсказано `negative`, скор `0.894`. Формально это сарказм, но в очищенном тексте остаются сильные негативные маркеры `впустую`, `бессмысленный`, `слабая`, поэтому кейс оказался проще, чем настоящий скрытый сарказм.
- Текст 4, короткий нейтральный/пограничный: ожидалось `neutral`, предсказано `neutral`, скор `0.547`. Ответ правильный, но уверенность самая низкая, что логично: текст короткий и почти без эмоциональных слов.
- Текст 5, эмоциональный негатив: ожидалось `negative`, предсказано `negative`, скор `0.926`. Это самый простой пример для модели, потому что негатив выражен напрямую словами `ужасно`, `мусор`, `плохой`, `провал`.


> **Вывод:**

На пяти новых текстах модель угадала все ожидаемые классы, лучше всего — прямой эмоциональный негатив (`0.926`), саркастически окрашенный негатив с явными плохими словами (`0.894`) и формальный негатив (`0.802`). Слабее всего она была уверена в коротком `neutral` (`0.547`) и разговорном `positive` (`0.626`).

Это подтверждает наблюдения из анализа ошибок: когда в тексте есть явные лексические маркеры тональности, TF-IDF + `LogisticRegression` работает вполне прилично. Но на коротких, нейтральных и по-настоящему смешанных отзывах модель остается осторожной, и для реального применения такие ответы лучше воспринимать как baseline, а не как финальную NLP-систему.


## Выводы

### 1. Обзор результатов

В работе использовался датасет из `131669` отзывов Кинопоиска с тремя классами: `positive` (`87138`), `neutral` (`24704`) и `negative` (`19827`). Данные загрузились аккуратно: после нормализации осталось `131669` строк и `6` столбцов, пустых отзывов и пустых `cleaned_text` не появилось. Предобработка через spaCy сохранила важные отрицания `не` и `нет`, что для тональности критично, потому что без них смысл многих фраз просто переворачивается.

EDA показал сильный дисбаланс в пользу `positive` и довольно длинные тексты: средняя длина `336.0` слов, медиана `288.0`, максимум `2059`. Лексика классов на сырых текстах сильно пересекается: в топах постоянно встречаются `фильм`, `фильма`, `все`, `очень`, `просто`, поэтому одной частотности слов недостаточно. В целом это объясняет, почему дальше важнее смотреть на `macro_f1`, а не только на `accuracy`.

### 2. Модели и качество

Лучшим baseline по `macro_f1` стала `LogisticRegression`: `macro_f1=0.6519`, `accuracy=0.7323`. `LinearSVC` после Optuna почти догнал ее по `macro_f1` (`0.6518`) и дал лучшую `accuracy=0.7714`, но по главной метрике все равно остался буквально на `0.000126` ниже. Поэтому итоговый выбор в ноутбуке — `LR baseline`: она проще, быстрее и не требует подбора гиперпараметров.

Optuna оказалась полезной не для всех одинаково. Для `LogisticRegression` прироста по `macro_f1` не вышло (`0.6519 → 0.6492`), для `LinearSVC` был заметный рост (`0.6343 → 0.6518`), а CatBoost даже после `hard`-подбора остался ниже линейных моделей. Сам CatBoost был полезен как обязательная третья модель и точка сравнения, но компромисс качество/время у него слабый: baseline обучался `119.60s` и дал только `macro_f1=0.6134`.

LSA тоже не улучшил результат: `TF-IDF + LSA` получил `macro_f1=0.6187` и `accuracy=0.6903`. Похоже, сжатие до `300` компонент убрало часть редких, но значимых маркеров тональности, поэтому прямой TF-IDF для этой задачи оказался практичнее.

### 3. Ошибки и дальнейшие шаги

У лучшей модели главная слабость — `neutral`. По confusion matrix она правильно распознала `2402` нейтральных отзыва из `4941`, а остальные часто ушли в `positive` (`1478`) или `negative` (`1061`). Самая массовая ошибка — `positive → neutral` (`2759` случаев). Это похоже на естественную проблему отзывов: часть текстов формально положительные, но написаны как пересказ сюжета или спокойное рассуждение.

На пяти новых текстах модель сработала правильно во всех случаях, но уверенность хорошо показывает ограничения: эмоциональный негатив получил `0.926`, формальный негатив `0.802`, а короткий нейтральный текст только `0.547`. В будущем я бы попробовал более сильную текстовую модель на эмбеддингах или RuBERT, отдельно поработал с `neutral` и добавил анализ ошибок по длине/уверенности. Сейчас результат выглядит как крепкий baseline, но не как финальная модель для тонких отзывов с сарказмом и смешанной оценкой.
